# DLPFC Low-Label Classification: PCA vs Image Embeddings

Same-slice classification with 25% / 50% / 75% labels, comparing PCA with PCA + image embeddings (GPFM / UNI / UNI2-h / GigaPath).


In [ ]:
import os, pickle, warnings, sys
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, matthews_corrcoef
import scanpy as sc
import torch
warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════
# Import core functions from the synthetic source files
# ══════════════════════════════════════════════════════════
# -- Clustering module: PCA / TruncatedSVD / StandardScaler
SPG_CLUSTER_DIR = Path("../../synthetic/src/05_downstream/spatial_cluster/DLPFC")
sys.path.insert(0, str(SPG_CLUSTER_DIR.resolve()))
import spagcn_cluster as spg

# -- Low-label classification module: classifier factory + training
SPG_LOWLABEL_DIR = Path("../../synthetic/src/05_downstream/supervised_low_label/DLPFC")
sys.path.insert(0, str(SPG_LOWLABEL_DIR.resolve()))
from classifiers import make_classifier as spg_make_classifier, train_classifier as spg_train_classifier
from predict import classifier_params as spg_classifier_params, transform_features as spg_transform_features, project_gpu_pca as spg_project_gpu_pca

# -- Data splitting module: low-label stratified sampling (identical to synthetic)
SPG_PREP_DIR = Path("../../synthetic/src/01_data_prep")
sys.path.insert(0, str(SPG_PREP_DIR.resolve()))
from prepare_low_label_dlpfc import make_low_label_split as spg_make_low_label_split

print("All dependencies imported successfully")
print(f"  classifiers.py: {SPG_LOWLABEL_DIR / 'classifiers.py'}")
print(f"  prepare_low_label_dlpfc.py: {SPG_PREP_DIR / 'prepare_low_label_dlpfc.py'}")


## Configuration

Parameters are read from `configs/supervised_low_label/DLPFC/downstream.yaml` and `configs/common/DLPFC/normalize.yaml`.


In [ ]:
# ══════════════════════════════════════════════════════════
# Read parameters from the synthetic config files
# ══════════════════════════════════════════════════════════
cfg = spg.load_config("configs/supervised_low_label/DLPFC/downstream.yaml")
task_cfg = cfg["supervised_low_label"]

# Slice IDs (reuse the spatial_cluster config)
sc_cfg = spg.load_config("configs/spatial_cluster/DLPFC/downstream.yaml")
SLICE_IDS = sorted(sc_cfg["task3_spatial_cluster"]["clustering"]["n_clusters_by_slice"].keys())

# Classification hyperparameters (from downstream.yaml)
SEEDS = task_cfg.get("seeds", [42, 43, 44])
PCA_N = task_cfg.get("pca_n_components", 50)
CLASSIFIER_NAMES = task_cfg.get("classifiers", ["LR", "XGBoost", "MLP"])

# Label fractions — actual values covered by the experiments (config default label_fraction: 0.50)
LABEL_FRACTIONS = [0.25, 0.50, 0.75]

# Metric list (from downstream.yaml metrics)
METRIC_CONFIG_NAMES = task_cfg.get("metrics", ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "mcc"])
METRIC_DISPLAY = {
    "accuracy": "Accuracy", "balanced_accuracy": "BalAcc",
    "macro_f1": "MacroF1", "weighted_f1": "WeightedF1", "mcc": "MCC",
}
METRIC_COLS = [METRIC_DISPLAY.get(m, m) for m in METRIC_CONFIG_NAMES]

# HVG parameters (from normalize.yaml)
hvg_cfg = spg.load_config("configs/common/DLPFC/normalize.yaml")
HVG_TOP_GENES = hvg_cfg.get("preprocessing", {}).get("DLPFC", {}).get("n_top_genes", 3000)

# Image embedding filename suffix
PKL_SUFFIX = "112"

# Data and output paths
DATA_DIR = Path("../../data/DLPFC")
OUTPUT_DIR = Path("../outputs/supervised_low_label_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Config: configs/supervised_low_label/DLPFC/downstream.yaml")
print(f"  Slices: {len(SLICE_IDS)}, seeds: {SEEDS}")
print(f"  PCA_N={PCA_N}, HVG_TOP_GENES={HVG_TOP_GENES}")
print(f"  Metrics: {METRIC_COLS}")
print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Output dir: {OUTPUT_DIR.resolve()}")


## Data Loading and Preprocessing

Use the synthetic preprocessed h5ad (QC + normalize + HVG) and align the image embeddings.


In [ ]:
# (key, display_name, dim) for image embeddings
EMBED_CONFIG = [
    ("gpfm",     "GPFM",   1024),
    ("uni",      "UNI",    1024),
    ("uni2_h",   "UNI2-h", 1536),
    ("gigapath", "GigaPath", 1536),
]

print("Loading embeddings...")
emb_data = {}
for key, name, dim in EMBED_CONFIG:
    emb_data[key] = {}
    for sid in SLICE_IDS:
        p = DATA_DIR / "pkl" / "embeddings" / f"visium_{sid}_allspot_{key}_{PKL_SUFFIX}.pkl"
        emb_data[key][sid] = torch.load(p, weights_only=True, map_location="cpu").numpy().astype("float32")
    print(f"  {name:20s} ({dim:4d}d): {len(emb_data[key])} slices")

# ══════════════════════════════════════════════════════════
# Data loading: use the h5ad preprocessed by synthetic
# First load the combined file to obtain the global shared genes (consistent with the synthetic pipeline)
# Then read slice by slice and align them all to that gene space
# ══════════════════════════════════════════════════════════
PROCESSED_DIR = Path("../../synthetic/data/02_interim/common/DLPFC")

# Load the combined file to obtain the global shared gene space (the synthetic pipeline uses all genes for PCA)
combined = ad.read_h5ad(PROCESSED_DIR / "processed_combined.h5ad")
# Use all shared genes (matching transform_features in predict.py)
COMMON_GENES = combined.var_names.tolist()
# To use the global HVG instead, change to: COMMON_GENES = combined.var_names[combined.var["highly_variable"]].tolist()
print(f"Global shared genes: {len(COMMON_GENES)}")

adatas = {}

for sid in SLICE_IDS:
    a = ad.read_h5ad(PROCESSED_DIR / f"{sid}_processed.h5ad")

    # Align embeddings: processed h5ad obs_names carry the "{sid}_" prefix
    a_raw = ad.read_h5ad(DATA_DIR / "st" / f"{sid}_adata.h5ad")
    raw_name_to_idx = {f"{sid}_{name}": i for i, name in enumerate(a_raw.obs_names)}
    keep_idx = [i for name, i in raw_name_to_idx.items() if name in set(a.obs_names.astype(str))]
    for key, _, _ in EMBED_CONFIG:
        e = emb_data.get(key, {}).get(sid)
        if e is not None and e.shape[0] != a.n_obs:
            emb_data[key][sid] = e[keep_idx]

    # Unify the gene space: take the global shared genes (replacing the previous per-slice HVG)
    # Ensure gene alignment when vstack-ing across slices, matching the synthetic pipeline
    common = [g for g in COMMON_GENES if g in a.var_names]
    a = a[:, common].copy()
    # Unify the label column name
    lbl = a.obs["label"]
    a.obs["spatialLIBD"] = lbl.values if hasattr(lbl, "values") else lbl
    a.obs["slice_id"] = sid
    adatas[sid] = a

print(f"Loaded {len(adatas)} slices, using {len(COMMON_GENES)} global shared genes (matching the synthetic pipeline)")


## Classifiers

LR / XGBoost / MLP from the synthetic `classifiers.py`.


In [ ]:
# The following functions are imported directly from synthetic:
#   spg_make_classifier(name, params, random_seed)  — classifiers.py
#   spg_train_classifier(x_train, y_train, classifier, params, random_seed) — classifiers.py
#   spg_classifier_params(classifier, use_gpu, n_jobs) — predict.py
#   spg_make_low_label_split(adata, label_fraction, seed, slice_col, label_col) — prepare_low_label_dlpfc.py

# Classifier parameters (identical to the synthetic predict.py)
CLF_PARAMS = {
    name: spg_classifier_params(name, use_gpu="auto", n_jobs=4)
    for name in CLASSIFIER_NAMES
}

print("Helper functions ready")
print("  make_classifier / train_classifier → from classifiers.py")
print("  classifier_params → from predict.py")
print("  make_low_label_split → from prepare_low_label_dlpfc.py (np.random.default_rng)")


## Feature Construction

PCA(50) baseline, and PCA(50) concatenated with each image embedding.


In [ ]:
def transform_pca_embedding_features(train_expr, test_expr, train_emb, test_emb, seed=42):
    """PCA+embedding: uses exactly the same underlying decomposition as the baseline.

    The baseline calls project_gpu_pca (torch eigh, exact decomposition) via
    predict.transform_features. Here we reuse that same function instead of
    starting a separate sklearn decomposition — empirically, even when
    svd_solver is explicitly set to "full", sklearn's SVD still differs from
    torch eigh on the later principal components, flipping ~2% of test spot predictions. Only the same implementation guarantees spot-by-spot consistency.
    """
    n = min(PCA_N, train_expr.shape[1], max(1, int(train_expr.shape[0]) - 1))
    if n < 2:
        return np.hstack([train_expr, train_emb]), np.hstack([test_expr, test_emb])
    tr = train_expr.toarray() if sparse.issparse(train_expr) else np.asarray(train_expr)
    te = test_expr.toarray() if sparse.issparse(test_expr) else np.asarray(test_expr)
    tr = np.asarray(tr, dtype=np.float32); te = np.asarray(te, dtype=np.float32)
    out = spg_project_gpu_pca(tr, tr, te, n, use_gpu="auto")
    if out is None:  # GPU unavailable: use the exact fallback equivalent to eigh
        pca = spg.PCA(n_components=n, random_state=42)  # matches the synthetic transform_features fallback
        pca.fit(tr)
        out = (pca.transform(tr), pca.transform(te))
    return np.hstack([out[0], train_emb]), np.hstack([out[1], test_emb])
print("Feature functions defined")
print("  transform_pca_features / transform_pca_embedding_features → match synthetic transform_features")


## Run the Experiments

For each (label fraction, classifier, seed): split, train, and evaluate.


In [ ]:
all_rows = []
pred_rows = []

for lf in LABEL_FRACTIONS:
    print(f"\n{'='*60}")
    print(f"Label fraction: {lf:.0%}")
    print(f"{'='*60}")
    for clf_name in CLASSIFIER_NAMES:
        for seed in SEEDS:
            # ── Consistent with the synthetic pipeline: use the same processed_combined.h5ad for the split ──
            # (prepare_low_label_dlpfc.py also calls make_low_label_split once on the combined data)
            combined_adata = ad.read_h5ad(PROCESSED_DIR / "processed_combined.h5ad")
            combined_adata.obs["spatialLIBD"] = combined_adata.obs["label"].values
            nan_mask = pd.isna(combined_adata.obs["spatialLIBD"])
            if nan_mask.any():
                combined_adata = combined_adata[~nan_mask].copy()
            combined_split, _ = spg_make_low_label_split(
                combined_adata, label_fraction=lf, seed=seed,
                slice_col="slice_id", label_col="spatialLIBD")

            for sid in SLICE_IDS:
                # Take the subset for the current slice from the combined split (obs order matches adatas[sid])
                slice_mask = combined_split.obs["slice_id"].astype(str).values == sid
                slice_split = combined_split[slice_mask].copy()
                train_mask = slice_split.obs["split"].astype(str).values == "train"
                test_mask = slice_split.obs["split"].astype(str).values == "test"
                if train_mask.sum() == 0 or test_mask.sum() == 0:
                    continue
                train_idx = np.where(train_mask)[0]
                test_idx = np.where(test_mask)[0]

                # Extract the train/test expression matrices
                X_train = slice_split.X[train_idx].toarray() if sparse.issparse(slice_split.X) else slice_split.X[train_idx]
                X_test = slice_split.X[test_idx].toarray() if sparse.issparse(slice_split.X) else slice_split.X[test_idx]

                # PCA only — fit on train, transform both
                train_adata = slice_split[train_mask]
                test_adata = slice_split[test_mask]

                train_labels_raw = slice_split.obs["spatialLIBD"].values[train_idx]
                test_labels_raw = slice_split.obs["spatialLIBD"].values[test_idx]

                feats_train, feats_test = spg_transform_features(train_adata, test_adata, train_adata, PCA_N, "auto")
                params = CLF_PARAMS[clf_name]
                _le = LabelEncoder()
                all_labels = np.concatenate([train_labels_raw, test_labels_raw])
                _le.fit(all_labels)
                train_labels_enc = _le.transform(train_labels_raw)
                clf = spg_train_classifier(feats_train, train_labels_enc,
                                           clf_name, params, random_seed=seed)
                pred = _le.inverse_transform(clf.predict(feats_test))
                true_test = test_labels_raw
                n_train = int(train_adata.n_obs)
                n_test = int(test_adata.n_obs)
                all_rows.append({"Slice": sid, "LabelFraction": lf, "Classifier": clf_name,
                    "Seed": seed, "Method": "PCA",
                    "Accuracy": round(accuracy_score(true_test, pred), 4),
                    "MacroF1": round(f1_score(true_test, pred, average="macro"), 4),
                    "WeightedF1": round(f1_score(true_test, pred, average="weighted"), 4),
                    "BalAcc": round(balanced_accuracy_score(true_test, pred), 4),
                    "MCC": round(matthews_corrcoef(true_test, pred), 4)})
                # --- Per-spot predictions (synthetic format) ---
                for obs_name, t, p in zip(test_adata.obs_names, true_test, pred):
                    pred_rows.append({
                        "obs_name": str(obs_name),
                        "slice_id": str(sid),
                        "y_true": str(t),
                        "y_pred": str(p),
                        "classifier": clf_name,
                        "model": "PCA",
                        "paradigm": "real_baseline",
                        "variant_id": "real_baseline",
                        "ratio": "",
                        "alpha": "",
                        "label_fraction": float(lf),
                        "low_label_seed": int(seed),
                        "n_train_real_slice": n_train,
                        "n_test_slice": n_test,
                        "n_train_package": n_train,
                        "regional_alpha_strategy": "",
                        "local_spatial_ratio_role": "",
                    })

                # PCA + each embedding — fit on train, transform both
                for k, name, _ in EMBED_CONFIG:
                    train_emb = emb_data[k][sid][train_idx]
                    test_emb = emb_data[k][sid][test_idx]
                    X_train_f = X_train
                    X_test_f = X_test
                    feats_g_train, feats_g_test = transform_pca_embedding_features(
                        X_train_f, X_test_f, train_emb, test_emb, seed)
                    clf = spg_train_classifier(feats_g_train, train_labels_enc,
                                               clf_name, params, random_seed=seed)
                    pred = _le.inverse_transform(clf.predict(feats_g_test))
                    all_rows.append({"Slice": sid, "LabelFraction": lf, "Classifier": clf_name,
                        "Seed": seed, "Method": f"PCA+{name}",
                        "Accuracy": round(accuracy_score(true_test, pred), 4),
                        "MacroF1": round(f1_score(true_test, pred, average="macro"), 4),
                        "WeightedF1": round(f1_score(true_test, pred, average="weighted"), 4),
                        "BalAcc": round(balanced_accuracy_score(true_test, pred), 4),
                        "MCC": round(matthews_corrcoef(true_test, pred), 4)})
                    # --- Per-spot predictions (synthetic format) ---
                    for obs_name, t, p in zip(test_adata.obs_names, true_test, pred):
                        pred_rows.append({
                            "obs_name": str(obs_name),
                            "slice_id": str(sid),
                            "y_true": str(t),
                            "y_pred": str(p),
                            "classifier": clf_name,
                            "model": f"PCA+{name}",
                            "paradigm": "real_baseline",
                            "variant_id": "real_baseline",
                            "ratio": "",
                            "alpha": "",
                            "label_fraction": float(lf),
                            "low_label_seed": int(seed),
                            "n_train_real_slice": n_train,
                            "n_test_slice": n_test,
                            "n_train_package": n_train,
                            "regional_alpha_strategy": "",
                            "local_spatial_ratio_role": "",
                        })
            print(f"  {clf_name} seed={seed} done", flush=True)

df = pd.DataFrame(all_rows)
print(f"\n{len(df)} result rows in total, {len(pred_rows)} per-spot predictions")

In [ ]:
metric_cols = METRIC_COLS

print("="*100)
print("  Global mean (across slices, seeds, and classifiers)")
print("="*100)
display(df.groupby(["LabelFraction", "Method", "Classifier"])[metric_cols].mean().round(4))

print("\n" + "="*100)
print("  Summary by label fraction")
print("="*100)
display(df.groupby(["LabelFraction", "Method"])[metric_cols].mean().round(4))

In [ ]:
# Delta (each embedding - PCA)
print("="*100)
print("  ΔAccuracy / ΔMacroF1 (each embedding - PCA)")
print("="*100)
for lf in LABEL_FRACTIONS:
    for clf_name in CLASSIFIER_NAMES:
        pca = df[(df["LabelFraction"]==lf) & (df["Method"]=="PCA") & (df["Classifier"]==clf_name)]
        if len(pca) == 0:
            continue
        baseline_acc = pca["Accuracy"].mean()
        baseline_f1 = pca["MacroF1"].mean()
        for method_name in [m for m in df["Method"].unique() if m != "PCA"]:
            sub = df[(df["LabelFraction"]==lf) & (df["Method"]==method_name) & (df["Classifier"]==clf_name)]
            if len(sub) == 0:
                continue
            da = sub["Accuracy"].mean() - baseline_acc
            df1 = sub["MacroF1"].mean() - baseline_f1
            print(f"  LabelFraction={lf:.0%}  {method_name:20s}  {clf_name:8s}:  ΔAcc={da:+.4f}  ΔMacroF1={df1:+.4f}")

In [ ]:
import json

df_pred = pd.DataFrame(pred_rows)
df_pred.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
print(f"Per-spot predictions saved: {OUTPUT_DIR / 'predictions.csv'} ({len(df_pred)} rows)")

# Also save the aggregated metrics for easy inspection
df.to_csv(OUTPUT_DIR / "per_run_results.csv", index=False)
df.groupby(["LabelFraction","Method","Classifier"])[metric_cols].agg(["mean","std"]).round(4).to_csv(OUTPUT_DIR / "summary.csv")

# ══════════════════════════════════════════════════════════════
# Save in a synthetic-compatible format:
# Output structure:
#   supervised_low_label/DLPFC/{model}/{fractionXX}/seed{seed}/
#     predictions.csv (subset for this method + fraction + seed)
#     manifest.json
# ══════════════════════════════════════════════════════════════
# model: PCA → "baseline", PCA+GPFM → "PCA_plus_GPFM", ...
# ══════════════════════════════════════════════════════════════
SPG_ROOT = Path("../outputs/supervised_low_label_comparison")
n_written = 0

for method_name in df["Method"].unique():
    is_pca = (method_name == "PCA")
    model_key = "baseline" if is_pca else method_name.replace("+", "_plus_")

    for (lf, seed), sub_preds in df_pred.groupby(["label_fraction", "low_label_seed"]):
        frac_key = f"fraction{int(float(lf)*100)}"
        seed_key = f"seed{int(seed)}"

        out_dir = SPG_ROOT / "supervised_low_label" / "DLPFC" / model_key / frac_key / seed_key
        out_dir.mkdir(parents=True, exist_ok=True)

        # ── predictions.csv ──
        sub = sub_preds[sub_preds["model"] == method_name].copy()
        sub.to_csv(out_dir / "predictions.csv", index=False)

        # ── manifest.json ──
        manifest = {
            "dataset": "DLPFC",
            "task_family": "supervised_low_label",
            "task_name": "dlpfc_slice_low_label_spot_clf",
            "model": model_key,
            "paradigm": "real_baseline",
            "variant_id": "baseline",
            "label_fraction": float(lf),
            "low_label_seed": int(seed),
            "classifiers": CLASSIFIER_NAMES,
        }
        (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        n_written += 1

print(f"\nResults saved to {OUTPUT_DIR.resolve()}/")
print(f"Wrote {n_written} (model, fraction, seed) directories in total, each with predictions.csv + manifest.json")